# Exploring the Security of One-Time and Many-Time Pads
**Applied Cryptography — Team Project**

Team members: *(fill in names)*

This notebook contains the code for all four problems:

| Problem | Content |
|---|---|
| 1 | One-time pad: definition, conditions, principles (written) |
| 2 | One-time pad: Alice (encrypt) and Bob (decrypt) programs |
| 3 | Many-time pad: 10 messages encrypted with one key, and the patterns that appear |
| 4 | Cryptanalysis: recovering plaintexts and the target message without the key |

## Problem 1 — Understanding the One-Time Pad

**Definition.** The one-time pad (OTP, Vernam cipher) is a symmetric cipher in which the plaintext is combined, symbol by symbol, with a secret key of the same length. For bits/bytes the combination is XOR ($\oplus$):

$$C = P \oplus K \qquad\qquad P = C \oplus K$$

Decryption works because $K \oplus K = 0$, so $(P \oplus K) \oplus K = P$.

**Conditions required for perfect secrecy**

1. **The key is truly random** – every key bit is uniform and independent of everything else (true random source or a cryptographically secure generator, never `random` or a predictable seed).
2. **The key is at least as long as the message** (normally exactly as long) – no key bits are recycled inside a message.
3. **The key is used only once** – a key must never encrypt a second message.
4. **The key stays secret** – only Alice and Bob hold it. It is exchanged securely in advance and destroyed after use.

**Why it is perfectly secure (Shannon).** For a ciphertext $c$ of length $n$ and *any* plaintext $m$ of length $n$ there is exactly one key, $k = m \oplus c$, that maps $m$ to $c$. All $2^{8n}$ keys are equally likely, so all plaintexts of that length are equally likely given $c$:

$$\Pr[P = m \mid C = c] = \Pr[P = m]$$

The ciphertext leaks nothing except the message length, even to an attacker with unlimited computing power.

**Practical limits.** The key is as long as all traffic, so key distribution is the hard part; OTP provides no integrity (flipping a ciphertext bit flips the same plaintext bit); and, as the Venona project showed, **reusing** pad material destroys the security guarantee (Problems 3 and 4).

## Problem 2 — One-Time Pad Implementation (Alice and Bob)

* **Alice** prompts for a plaintext, draws a fresh random key of the same length with `secrets.token_bytes` (a CSPRNG), prints the ciphertext and saves ciphertext and key (hex) in two separate files.
* **Bob** reads the two files and prints the recovered plaintext.

In [1]:
import os
import secrets

def xor_bytes(data: bytes, key: bytes) -> bytes:
    """XOR two byte strings of identical length (the core OTP operation)."""
    if len(data) != len(key):
        raise ValueError("data and key must have the same length")
    return bytes(d ^ k for d, k in zip(data, key))


def alice(cipher_file: str = "ciphertext.hex", key_file: str = "key.hex") -> None:
    """Alice: read a plaintext, encrypt it with a fresh random key, save both to files."""
    plaintext = input("Alice - enter your message: ").encode("utf-8")

    # Fresh, uniformly random key, exactly as long as the message.
    key = secrets.token_bytes(len(plaintext))
    ciphertext = xor_bytes(plaintext, key)

    print("Ciphertext (hex):", ciphertext.hex())

    # Ciphertext and key go into SEPARATE files (the ciphertext may be sent over
    # an insecure channel; the key must travel/stay secret).
    with open(cipher_file, "w") as f:
        f.write(ciphertext.hex())
    with open(key_file, "w") as f:
        f.write(key.hex())
    print(f"Saved ciphertext to '{cipher_file}' and key to '{key_file}'.")

In [2]:
def bob(cipher_file: str = "ciphertext.hex", key_file: str = "key.hex") -> str:
    """Bob: read key and ciphertext from their files and recover the plaintext."""
    with open(key_file) as f:
        key = bytes.fromhex(f.read().strip())
    with open(cipher_file) as f:
        ciphertext = bytes.fromhex(f.read().strip())

    plaintext = xor_bytes(ciphertext, key).decode("utf-8")
    print("Bob decrypted:", plaintext)
    return plaintext

In [3]:
# Run Alice, then Bob. (Type any message when prompted.)
alice()
recovered = bob()

print("\nFiles created:", [f for f in ("ciphertext.hex", "key.hex") if os.path.exists(f)])
print("key.hex       :", open("key.hex").read())
print("ciphertext.hex:", open("ciphertext.hex").read())

Alice - enter your message: Hello Bob, meet me at the old bridge at midnight.
Ciphertext (hex): 6fb4842672a21b192227c18bab25309e67d60f4ca1c4d7155da3b5ca691a71e18f7ee8a1b9948d73c7969aed505ca156fc
Saved ciphertext to 'ciphertext.hex' and key to 'key.hex'.
Bob decrypted: Hello Bob, meet me at the old bridge at midnight.

Files created: ['ciphertext.hex', 'key.hex']
key.hex       : 27d1e84a1d825976400be1e6ce4044be0ab32f2dd5e4a37d3883daa60d3a1393e61a8fc499f5f953aafffe83393bc922d2
ciphertext.hex: 6fb4842672a21b192227c18bab25309e67d60f4ca1c4d7155da3b5ca691a71e18f7ee8a1b9948d73c7969aed505ca156fc


## Problem 3 — Many-Time Pad (one key, ten messages)

We now reuse **one** key for ten predefined messages. The key is made long enough for the longest one. The plaintexts, the key and the ciphertexts (all in hex) are written to a file. Then we look for patterns in the outputs, which Problem 4 will exploit.

In [4]:
MESSAGES = [
    "The one-time pad is perfectly secure only when the key is random, secret, as long as the message and never reused.",
    "During the Second World War Soviet intelligence services reused some of their pages, and analysts later exploited that mistake.",
    "Meet me tomorrow at the old library near the river, and please bring along the documents we discussed last week.",
    "A stream cipher XORs a keystream with the plaintext, so reusing the same keystream leaks the XOR of two messages.",
    "Never trust a system whose security depends on the secrecy of its design rather than on the secrecy of its key.",
    "The attacker does not need the key at all, because XORing two ciphertexts cancels the key and exposes both messages.",
    "Cryptography is hard to get right, and the smallest implementation mistake can quietly destroy an otherwise perfect scheme.",
    "Please confirm that the package arrived safely at the harbor and that nobody followed you on the way to the station.",
    "Spaces are the weakness here, since a space XORed with a letter simply flips the case of that letter in the result.",
    "Always generate fresh random keys from a cryptographically secure source, and destroy every key right after using it.",
]

def many_time_pad(messages, key=None):
    """Encrypt every message with the SAME key (insecure on purpose)."""
    plaintexts = [m.encode("utf-8") for m in messages]
    if key is None:
        key = secrets.token_bytes(max(len(p) for p in plaintexts))   # long enough for all
    ciphertexts = [xor_bytes(p, key[:len(p)]) for p in plaintexts]   # same key, every time
    return plaintexts, key, ciphertexts


def save_many_time_pad(plaintexts, key, ciphertexts, filename="many_time_pad.txt"):
    """Write plaintexts, key and ciphertexts (all hex) into one file."""
    with open(filename, "w") as f:
        f.write(f"KEY: {key.hex()}\n\n")
        for i, (p, c) in enumerate(zip(plaintexts, ciphertexts), start=1):
            f.write(f"PLAINTEXT_{i}: {p.hex()}\n")
            f.write(f"CIPHERTEXT_{i}: {c.hex()}\n\n")


plaintexts, key, ciphertexts = many_time_pad(MESSAGES)
save_many_time_pad(plaintexts, key, ciphertexts)

print(f"Key length: {len(key)} bytes; message lengths: {[len(p) for p in plaintexts]}")
print("First ciphertext (hex):", ciphertexts[0].hex()[:64], "...")

Key length: 127 bytes; message lengths: [114, 127, 112, 113, 111, 116, 123, 116, 115, 117]
First ciphertext (hex): 4a1dafe98beb66464cda9a660a2d3ec66d889c47a44f15621eea124799c8fc01 ...


### Observations: patterns in the many-time pad outputs

In [5]:
import itertools

def printable(b: bytes) -> str:
    """Show a byte string with non-printable bytes as '.'."""
    return "".join(chr(x) if 32 <= x < 127 else "." for x in b)

# Observation 1: the key cancels out:  C_i xor C_j == P_i xor P_j  (for ALL 45 pairs)
all_ok = True
for i, j in itertools.combinations(range(10), 2):
    n = min(len(ciphertexts[i]), len(ciphertexts[j]))
    all_ok &= xor_bytes(ciphertexts[i][:n], ciphertexts[j][:n]) == xor_bytes(plaintexts[i][:n], plaintexts[j][:n])
print("Obs 1: C_i xor C_j == P_i xor P_j for all 45 pairs:", all_ok)

# Observation 2: changing the KEY does not change C_i xor C_j at all.
_, key2, cts2 = many_time_pad(MESSAGES)           # brand-new random key
n = min(len(ciphertexts[0]), len(ciphertexts[1]))
same = xor_bytes(ciphertexts[0][:n], ciphertexts[1][:n]) == xor_bytes(cts2[0][:n], cts2[1][:n])
print("Obs 2: new key => completely different ciphertexts, but C1 xor C2 unchanged:", same)

# Observation 3: changing ONE plaintext changes only the matching ciphertext bytes.
altered = list(MESSAGES); altered[0] = "XHE" + MESSAGES[0][3:]       # change the first 3 chars only
_, _, cts3 = many_time_pad(altered, key=key)
diff_positions = [i for i in range(len(ciphertexts[0])) if ciphertexts[0][i] != cts3[0][i]]
print("Obs 3: positions where ciphertext 1 changed:", diff_positions,
      "| other ciphertexts unchanged:", all(ciphertexts[k] == cts3[k] for k in range(1, 10)))

# Observation 4: equal plaintext bytes at the same position give 0x00.
x = xor_bytes(ciphertexts[0][:n], ciphertexts[1][:n])
print("Obs 4: zero bytes in C1 xor C2 at positions:", [i for i, b in enumerate(x) if b == 0],
      "-> P1/P2 agree there:", [chr(plaintexts[0][i]) for i, b in enumerate(x) if b == 0])

# Observation 5: space (0x20) xor letter = the same letter with its case flipped.
print("\nObs 5: letters in C3 xor C4 mark positions where one message has a SPACE")
n34 = min(len(ciphertexts[2]), len(ciphertexts[3]))
x34 = xor_bytes(ciphertexts[2][:n34], ciphertexts[3][:n34])
print("P3      :", MESSAGES[2][:60])
print("P4      :", MESSAGES[3][:60])
print("C3^C4   :", printable(x34)[:60])
print("(a letter at position i means a space was opposite a letter; the letter shown is the OTHER message's letter, case flipped)")

Obs 1: C_i xor C_j == P_i xor P_j for all 45 pairs: True
Obs 2: new key => completely different ciphertexts, but C1 xor C2 unchanged: True
Obs 3: positions where ciphertext 1 changed: [0, 1, 2] | other ciphertexts unchanged: True
Obs 4: zero bytes in C1 xor C2 at positions: [39, 49, 67, 80, 84, 88, 104, 111, 112] -> P1/P2 agree there: ['l', 'e', 'e', 'g', ' ', ' ', 'e', 'e', 'd']

Obs 5: letters in C3 xor C4 mark positions where one message has a SPACE
P3      : Meet me tomorrow at the old library near the river, and plea
P4      : A stream cipher XORs a keystream with the plaintext, so reus
C3^C4   : .E..R..MT......Wx.&ST.EK...T....A..T.E..ET..A.....X.A.......
(a letter at position i means a space was opposite a letter; the letter shown is the OTHER message's letter, case flipped)


**What we learn:** the key cancels out (Obs 1, 2), so the attacker obtains the XOR of two plaintexts without any key; equal bytes produce zero bytes (Obs 4); and, because English text is full of spaces, a space opposite a letter shows up as a readable, case-flipped letter (Obs 5). Plaintext structure therefore leaks directly through the ciphertexts. Problem 4 turns this into a full attack.

In [ ]:
import json
import math
import string
import numpy as np

# ---------- 0. Load the challenge ----------
CIPHERTEXTS_HEX = []    # <- paste the instructor's ciphertexts here (hex strings), or use challenge.json
TARGET_HEX = ""         # <- paste the target ciphertext here (hex string)

if os.path.exists("challenge.json"):
    with open("challenge.json") as f:
        ch = json.load(f)
    CIPHERTEXTS_HEX, TARGET_HEX = ch["ciphertexts"], ch["target"]

if CIPHERTEXTS_HEX and TARGET_HEX:
    challenge_cts = [bytes.fromhex(h) for h in CIPHERTEXTS_HEX]
    target_ct = bytes.fromhex(TARGET_HEX)
    TRUTH_KEY = None                                   # real challenge: no ground truth available
    print("Loaded external challenge.")
else:
    # Self-made challenge: the Problem 3 ciphertexts plus an 11th (target) message, SAME key.
    TARGET_PLAINTEXT = "The secret message is: when using a stream cipher, never use the key more than once!"
    target_ct = xor_bytes(TARGET_PLAINTEXT.encode(), key[:len(TARGET_PLAINTEXT)])
    challenge_cts = list(ciphertexts)
    TRUTH_KEY = key
    print("Built demo challenge from Problem 3 (ground truth available for verification).")

print(f"{len(challenge_cts)} ciphertexts, lengths {[len(c) for c in challenge_cts]}, target length {len(target_ct)}")

Built demo challenge from Problem 3 (ground truth available for verification).
10 ciphertexts, lengths [114, 127, 112, 113, 111, 116, 123, 116, 115, 117], target length 84


In [7]:
# ---------- 1. English character model ----------
# Approximate letter frequencies of English (percent), most frequent first.
LETTER_FREQ = dict(zip("etaoinshrdlcumwfgypbvkjxqz",
    [12.7, 9.1, 8.2, 7.5, 7.0, 6.7, 6.3, 6.1, 6.0, 4.3, 4.0, 2.8, 2.8,
     2.4, 2.4, 2.2, 2.0, 2.0, 1.9, 1.5, 1.0, 0.8, 0.15, 0.15, 0.1, 0.07]))

def build_log_model() -> np.ndarray:
    """Return an array LOG[b] = log P(byte b appears in English text)."""
    prob = {}
    total = sum(LETTER_FREQ.values())
    for ch, f in LETTER_FREQ.items():
        prob[ch] = 0.74 * f / total            # ~74% of characters: lower-case letters
        prob[ch.upper()] = 0.02 * f / total    # ~2%: capitals
    prob[" "] = 0.17                           # the space is the most frequent character
    for ch in ",.":                 prob[ch] = 0.012
    for ch in "'-:;!?\"()":         prob[ch] = 0.002
    for ch in string.digits:        prob[ch] = 0.001
    floor = 1e-7                               # anything else (unprintable bytes, odd symbols)
    return np.array([math.log(prob.get(chr(b), floor)) for b in range(256)])

LOG = build_log_model()
GUESSES = np.arange(256, dtype=np.uint8)       # all possible key bytes


def recover_key(ciphertexts) -> bytes:
    """Column-wise attack: for every position pick the key byte that makes the
    whole column of ciphertext bytes look most like English."""
    length = max(len(c) for c in ciphertexts)
    key_guess = bytearray(length)
    for i in range(length):
        column = np.array([c[i] for c in ciphertexts if len(c) > i], dtype=np.uint8)
        # scores[g] = sum_j log P(c_j[i] xor g)
        scores = LOG[column[:, None] ^ GUESSES[None, :]].sum(axis=0)
        key_guess[i] = int(np.argmax(scores))
    return bytes(key_guess)


def show(label: str, ct: bytes, k: bytes) -> None:
    n = min(len(ct), len(k))
    print(f"{label:<10}{printable(xor_bytes(ct[:n], k[:n]))}")

In [8]:
# ---------- 2. Fully automatic attack (no human involved) ----------
# The target is part of the column set, so it also helps to determine the key.
auto_key = recover_key(challenge_cts + [target_ct])

print("AUTOMATIC RESULT")
for i, c in enumerate(challenge_cts, 1):
    show(f"msg {i}", c, auto_key)
print()
show("TARGET", target_ct, auto_key)

if TRUTH_KEY is not None:
    ok = sum(a == b for a, b in zip(auto_key, TRUTH_KEY))
    print(f"\nKey bytes recovered exactly: {ok}/{len(TRUTH_KEY)}")

AUTOMATIC RESULT
msg 1     the one-time pad is perfectly secure only when the key is random, secret, as long ar the message and neven reused.
msg 2     during the Second World War Soviet intelligence services reused some of their pages- and analysts later edploited thar aee.    
msg 3     meet me tomorrow at the old library near the river, and please bring along the doculents we discussed lash week.
msg 4     a stream cipher XORs a keystream with the plaintext, so reusing the same keystream meaks the XOR of two myssages.
msg 5     never trust a system whose security depends on the secrecy of its design rather thao on the secrecy of ito key.
msg 6     the attacker does not need the key at all, because XORing two ciphertexts cancels tie key and exposes bott messages.
msg 7     cryptography is hard to get right, and the smallest implementation mistake can quieuly destroy an otherwioe perfect sehiast
msg 8     please confirm that the package arrived safely at the harbor and that nobody fol

The automatic stage already reads almost everything. Typical remaining defects: the **first letter** of every message is lower-case (capital vs. lower-case differ by a single bit, and lower-case is statistically more likely), a few **rare punctuation** characters, and a short **tail** that is covered by only one or two ciphertexts, so there is too little evidence. A human reading the output can repair these with cribs.

In [9]:
# ---------- 3. Human-in-the-loop refinement with cribs ----------
def apply_crib(key_guess: bytearray, ct: bytes, pos: int, crib: str) -> None:
    """Assume ciphertext `ct` contains `crib` starting at `pos`; derive the key bytes
    k[i] = c[i] xor crib[i] and write them into the working key."""
    crib_b = crib.encode()
    key_guess[pos:pos + len(crib_b)] = xor_bytes(ct[pos:pos + len(crib_b)], crib_b)


def crib_drag(ciphertexts, idx: int, crib: str, top: int = 3):
    """Slide `crib` along ciphertext `idx`. For each offset the implied key bytes are
    used to decrypt the OTHER ciphertexts; offsets that make them all look like
    English (high average log-probability) are the likely positions of the crib."""
    crib_b, target = crib.encode(), ciphertexts[idx]
    results = []
    for off in range(len(target) - len(crib_b) + 1):
        implied = xor_bytes(target[off:off + len(crib_b)], crib_b)
        scores = [LOG[b] for j, c in enumerate(ciphertexts) if j != idx and len(c) >= off + len(crib_b)
                  for b in xor_bytes(c[off:off + len(crib_b)], implied)]
        if scores:
            results.append((sum(scores) / len(scores), off))
    return sorted(results, reverse=True)[:top]


# (a) Where is the word "message" in the target?  Crib-drag it.
for score, off in crib_drag(challenge_cts + [target_ct], len(challenge_cts), "message"):
    print(f"offset {off:3d}  avg log-prob {score:6.2f}")

offset  11  avg log-prob  -3.04
offset   6  avg log-prob  -5.08
offset   2  avg log-prob  -5.31


In [10]:
# (b) Apply the human corrections after reading the automatic output above.
# Each entry is (position in the target, word we are confident about).
# For the demo challenge these are: the capital "T" of "The" and the final "once!".
# For the instructor's challenge, read the automatic output and edit this list.
if TRUTH_KEY is not None:
    HUMAN_CRIBS = [(0, "The"), (len(target_ct) - 5, "once!")]
else:
    HUMAN_CRIBS = []          # <- add your own (position, crib) guesses here

working_key = bytearray(auto_key)
for pos, crib in HUMAN_CRIBS:
    apply_crib(working_key, target_ct, pos, crib)

print("AFTER HUMAN CORRECTIONS")
show("TARGET", target_ct, working_key)

recovered_target = xor_bytes(target_ct, bytes(working_key[:len(target_ct)])).decode("utf-8", errors="replace")
print("\nRecovered target message:\n ", recovered_target)

if TRUTH_KEY is not None:
    print("Matches the real target plaintext:", recovered_target == TARGET_PLAINTEXT)

AFTER HUMAN CORRECTIONS
TARGET    The secret message is: when using a stream cipher, never use the key more than once!

Recovered target message:
  The secret message is: when using a stream cipher, never use the key more than once!
Matches the real target plaintext: True


### Analysis of the attack
How reliable is the automatic stage, and what does it depend on? We repeat the attack with fresh random keys on subsets of $n$ of the eleven messages and measure the fraction of characters recovered (ignoring upper/lower case, which is the one systematic ambiguity).

In [11]:
import random

POOL = [m.encode() for m in MESSAGES] + [TARGET_PLAINTEXT.encode()] if TRUTH_KEY is not None else None

def accuracy_for(n: int, trials: int = 25) -> float:
    """Average fraction of plaintext characters recovered (case-insensitive)
    when n messages share one random key."""
    total = correct = 0
    for _ in range(trials):
        subset = random.sample(POOL, n)
        k = secrets.token_bytes(max(len(p) for p in subset))
        cts = [xor_bytes(p, k[:len(p)]) for p in subset]
        guess = recover_key(cts)
        for p, c in zip(subset, cts):
            dec = xor_bytes(c, guess[:len(c)]).lower()
            correct += sum(a == b for a, b in zip(dec, p.lower()))
            total += len(p)
    return correct / total

if POOL is not None:
    random.seed(1)
    print(" n ciphertexts | characters recovered")
    for n in (2, 3, 4, 5, 7, 9, 11):
        print(f"{n:>13} | {100 * accuracy_for(n):6.1f}%")

 n ciphertexts | characters recovered
            2 |   28.9%
            3 |   58.0%
            4 |   72.0%
            5 |   80.1%
            7 |   91.2%
            9 |   95.3%
           11 |   97.3%


In [12]:
# Coverage: a key byte can only be recovered well where several ciphertexts overlap.
lengths = sorted((len(c) for c in challenge_cts + [target_ct]), reverse=True)
longest = lengths[0]
for depth in (1, 2, 3, 5, 8):
    covered = sum(1 for i in range(longest) if sum(l > i for l in lengths) >= depth)
    print(f"positions covered by >= {depth:>2} ciphertexts: {covered:>3}/{longest}")

positions covered by >=  1 ciphertexts: 127/127
positions covered by >=  2 ciphertexts: 123/127
positions covered by >=  3 ciphertexts: 117/127
positions covered by >=  5 ciphertexts: 116/127
positions covered by >=  8 ciphertexts: 113/127


### Conclusions

* **Key reuse destroys the one-time pad.** The key cancels in $C_i \oplus C_j = P_i \oplus P_j$, turning the problem into breaking a "two-text" cipher, which structured language (spaces, frequent letters) makes easy; the more ciphertexts, the easier it gets (table above).
* The attack needs **no key and no brute force** — only statistics per column plus a few human guesses.
* Weak spots: the beginnings/ends of messages, where there is less evidence (the tail beyond the second-longest message is covered by a single ciphertext), and case/punctuation ambiguities.
* **Defence:** never reuse a key (use a fresh, uniformly random key per message), or use a stream cipher with a unique nonce per message (e.g. ChaCha20, AES-CTR) together with an authenticated mode. This is the lesson of Venona: a few duplicated pads were enough for cryptanalysts to read Soviet traffic for years.